In [1]:
import nest_asyncio
import os
import pickle
from llama_index.core import VectorStoreIndex

In [2]:
with open("embeddings.pkl", "rb") as f:
    embeds = pickle.load(f)

In [3]:
from pinecone.grpc import PineconeGRPC
from llama_index.vector_stores.pinecone import PineconeVectorStore
from llama_index.core import StorageContext

In [15]:
from dotenv import load_dotenv
from pinecone import ServerlessSpec
load_dotenv()
api = os.getenv("PINECONE_API_KEY")

pine = PineconeGRPC(api_key=api)
pine.create_index("embeddings", dimension=1024, metric='cosine', spec=ServerlessSpec(cloud="aws",region="us-east-1"))

PineconeApiException: (409)
Reason: Conflict
HTTP response headers: HTTPHeaderDict({'content-type': 'text/plain; charset=utf-8', 'vary': 'origin, access-control-request-method, access-control-request-headers', 'access-control-allow-origin': '*', 'access-control-expose-headers': '*', 'x-pinecone-api-version': '2025-04', 'x-cloud-trace-context': 'd240e926e3513b111a205b456dc13667', 'date': 'Mon, 03 Aug 2026 13:56:19 GMT', 'server': 'Google Frontend', 'Content-Length': '85', 'Via': '1.1 google', 'Alt-Svc': 'h3=":443"; ma=2592000'})
HTTP response body: {"error":{"code":"ALREADY_EXISTS","message":"Resource  already exists"},"status":409}


In [23]:
with open("extracts.pkl", "rb") as f:
    nodes = pickle.load(f)

In [16]:
index = pine.Index("embeddings")

In [17]:
vectorstore = PineconeVectorStore(pinecone_index=index)

In [27]:
vectors=[]
for i, embed in enumerate(embeds):
    nodal = {
        "id" : f"{i}",
        "values": embed,
        "metadata": { "text": nodes[i].text, **nodes[i].metadata}
        
    }
    vectors.append(nodal)


In [32]:
for i in range(0,len(vectors), 100):
    batch = vectors[i:i+100]
    if i == 9500:
        batch = vectors[i:]
    index.upsert(vectors=batch)
    print("upserted ",i)

upserted  0
upserted  100
upserted  200
upserted  300
upserted  400
upserted  500
upserted  600
upserted  700
upserted  800
upserted  900
upserted  1000
upserted  1100
upserted  1200
upserted  1300
upserted  1400
upserted  1500
upserted  1600
upserted  1700
upserted  1800
upserted  1900
upserted  2000
upserted  2100
upserted  2200
upserted  2300
upserted  2400
upserted  2500
upserted  2600
upserted  2700
upserted  2800
upserted  2900
upserted  3000
upserted  3100
upserted  3200
upserted  3300
upserted  3400
upserted  3500
upserted  3600
upserted  3700
upserted  3800
upserted  3900
upserted  4000
upserted  4100
upserted  4200
upserted  4300
upserted  4400
upserted  4500
upserted  4600
upserted  4700
upserted  4800
upserted  4900
upserted  5000
upserted  5100
upserted  5200
upserted  5300
upserted  5400
upserted  5500
upserted  5600
upserted  5700
upserted  5800
upserted  5900
upserted  6000
upserted  6100
upserted  6200
upserted  6300
upserted  6400
upserted  6500
upserted  6600
upserte

In [33]:
from llama_index.llms.groq import Groq

In [34]:
api = os.getenv("GROQ_API_KEY")
llm = Groq(model="openai/gpt-oss-120b", api=api)

In [45]:
from llama_index.core.retrievers import VectorIndexRetriever
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core import get_response_synthesizer

In [47]:
retriver = VectorIndexRetriever(index=vectorstore,similarity_top_k=10 )
synth = get_response_synthesizer()
query_engine = RetrieverQueryEngine(retriever=retriver, response_synthesizer=synth)
response = query_engine.query("Outline some techniques of conservation ")

AttributeError: 'PineconeVectorStore' object has no attribute 'vector_store'

In [58]:
from llama_index.embeddings.huggingface_api import HuggingFaceInferenceAPIEmbedding
import nest_asyncio

nest_asyncio.apply()
embed_model = HuggingFaceInferenceAPIEmbedding(
    model_name="BAAI/bge-m3",
    token=os.getenv("HF_API_KEY"),
)

In [66]:
query_embed = embed_model.get_text_embedding("provide white tiger conservation strategies")

In [ ]:
response = index.query(
    vector=query_embed,
    top_k=10,
    include_metadata=True
)
print(response)

{'matches': [{'id': '9335',
              'metadata': {'creation_date': '2026-07-19',
                           'excerpt_keywords': 'ESA, Endangered, Recovery, '
                                               'White, Strategies\n'
                                               '\n'
                                               'The provided text is a',
                           'file_name': 'improving-the-effectiveness-and-efficiency-of-the-endangered-species-act.pdf',
                           'file_path': '/Users/rifah/Desktop/p/species/docs/improving-the-effectiveness-and-efficiency-of-the-endangered-species-act.pdf',
                           'file_size': 1114002.0,
                           'file_type': 'application/pdf',
                           'last_modified_date': '2026-07-19',
                           'organizations': ['ESA'],
                           'page_label': '10',
                           'text': 'DeSert tortoiSe reCovery oFFi Ce, U.S. '
                 